In [163]:
from dotenv import load_dotenv


load_dotenv(".env")


True

In [164]:
from dotenv import load_dotenv
import os
import sounddevice as sd
from scipy.io.wavfile import write
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Check your .env file.")

client = OpenAI(api_key=OPENAI_API_KEY)

In [82]:
import os
import json
import base64
import threading
import time
from pathlib import Path

import sounddevice as sd
import websocket
from dotenv import load_dotenv
from openai import OpenAI

# Works whether the notebook is run from the repo root, codebase/, or codebase/notebook/.
for env_path in (Path(".env"), Path("../.env"), Path("codebase/.env"), Path("codebase/notebook/.env")):
    if env_path.exists():
        load_dotenv(env_path)
        print(f"Loaded env: {env_path.resolve()}")
        break
else:
    load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Check your .env file location and variable name.")

SAMPLE_RATE = 24000
CHANNELS = 1
BLOCK_DURATION = 0.1
BLOCK_SIZE = int(SAMPLE_RATE * BLOCK_DURATION)
SILENCE_THRESHOLD = 300
SILENCE_SECONDS = 5.0
MAX_RECORD_SECONDS = 300

TRANSCRIPTION_MODEL = "gpt-4o-mini-transcribe"
CLOSE_AFTER_FIRST_FINAL = False
print("Realtime transcription cell v2")

# Preflight turns auth/model/access problems into readable errors before WebSocket starts.
client = OpenAI(api_key=OPENAI_API_KEY)
try:
    client.models.retrieve(TRANSCRIPTION_MODEL)
    print(f"Transcription model accessible: {TRANSCRIPTION_MODEL}")
except Exception as exc:
    raise RuntimeError(
        f"Could not access {TRANSCRIPTION_MODEL}. Check API key, project billing, and model access."
    ) from exc

url = "wss://api.openai.com/v1/realtime?intent=transcription"
headers = [
    "Authorization: Bearer " + OPENAI_API_KEY,
    "OpenAI-Safety-Identifier: local-notebook-user",
]

stop_event = threading.Event()
latest_final_transcript = ""

def on_open(ws):
    print("Connected. Start speaking...")

    ws.send(json.dumps({
        "type": "session.update",
        "session": {
            "type": "transcription",
            "audio": {
                "input": {
                    "format": {
                        "type": "audio/pcm",
                        "rate": SAMPLE_RATE
                    },
                    "transcription": {
                        "model": TRANSCRIPTION_MODEL,
                        "language": "en"
                    },
                    "turn_detection": None
                }
            }
        }
    }))

    def stream_microphone():
        silent_chunks = 0
        segment_chunks = 0
        total_chunks = 0
        speech_detected = False
        started_at = time.monotonic()
        last_status_at = 0

        try:
            with sd.InputStream(
                samplerate=SAMPLE_RATE,
                channels=CHANNELS,
                dtype="int16",
                blocksize=BLOCK_SIZE,
            ) as stream:
                while not stop_event.is_set() and ws.sock and ws.sock.connected:
                    audio_chunk, _ = stream.read(BLOCK_SIZE)
                    segment_chunks += 1
                    total_chunks += 1

                    rms = float((audio_chunk.astype("float32") ** 2).mean() ** 0.5)
                    if rms > SILENCE_THRESHOLD:
                        speech_detected = True
                        silent_chunks = 0
                    elif speech_detected:
                        silent_chunks += 1

                    audio_base64 = base64.b64encode(audio_chunk.tobytes()).decode("utf-8")
                    ws.send(json.dumps({
                        "type": "input_audio_buffer.append",
                        "audio": audio_base64
                    }))

                    silence_time = silent_chunks * BLOCK_DURATION
                    segment_time = segment_chunks * BLOCK_DURATION
                    total_time = time.monotonic() - started_at
                    should_commit = speech_detected and silence_time >= SILENCE_SECONDS
                    reached_limit = total_time >= MAX_RECORD_SECONDS

                    if total_time - last_status_at >= 1:
                        print(
                            f"\rTotal: {total_time:6.1f}s | Segment: {segment_time:5.1f}s | Pause: {silence_time:4.1f}s",
                            end="",
                            flush=True,
                        )
                        last_status_at = total_time

                    if should_commit:
                        print("\nFinalizing audio segment...")
                        ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
                        silent_chunks = 0
                        segment_chunks = 0
                        speech_detected = False

                    elif reached_limit:
                        print("\nMax recording time reached. Finalizing audio...")
                        ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
                        stop_event.set()
                        return
        except Exception as exc:
            if not stop_event.is_set():
                print("\nMicrophone stream stopped:", repr(exc))
                ws.close()

    threading.Thread(target=stream_microphone, daemon=True).start()

def on_message(ws, message):
    global latest_final_transcript

    event = json.loads(message)
    event_type = event.get("type")

    if event_type == "session.updated":
        print("Session ready.")

    elif event_type == "conversation.item.input_audio_transcription.delta":
        print(event.get("delta", ""), end="", flush=True)

    elif event_type == "conversation.item.input_audio_transcription.completed":
        latest_final_transcript = event.get("transcript", "")
        print("\nFinal:", latest_final_transcript)

        if CLOSE_AFTER_FIRST_FINAL:
            stop_event.set()
            ws.close()

    elif event_type == "error":
        print("\nError:", event)
        stop_event.set()
        ws.close()

def on_error(ws, error):
    print("WebSocket error:", error)
    stop_event.set()

def on_close(ws, close_status_code, close_msg):
    stop_event.set()
    print(f"\nConnection closed. code={close_status_code}, message={close_msg}")

ws = websocket.WebSocketApp(
    url,
    header=headers,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
)

try:
    ws.run_forever(ping_interval=20, ping_timeout=10)
except KeyboardInterrupt:
    stop_event.set()
    ws.close()
    print("Stopped by user.")


Loaded env: D:\AI PRODUCTS\TODO LIST - AI ASSISTED[ VOICE AGENT]\codebase\.env
Realtime transcription cell v2
Transcription model accessible: gpt-4o-mini-transcribe
Connected. Start speaking...
Session ready.
Total:    9.6s | Segment:   9.5s | Pause:  0.0sWebSocket error: 

Connection closed. code=None, message=None


In [165]:
latest_final_transcript="Final: Hi, today is the 2026 May 16th. So my plan for today is I'm going to first develop the Langgraph-powered AI voice assistant system, which will take to-do list as audio in audio format, and then Langgraph will process that input, which will be converted to text before feeding into Langgraph. Then Langgraph will process that, prioritize that, and create the to-do list."
latest_final_transcript

"Final: Hi, today is the 2026 May 16th. So my plan for today is I'm going to first develop the Langgraph-powered AI voice assistant system, which will take to-do list as audio in audio format, and then Langgraph will process that input, which will be converted to text before feeding into Langgraph. Then Langgraph will process that, prioritize that, and create the to-do list."

In [166]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage

llm=ChatOpenAI(model="gpt-5.4-mini")

agent=create_agent(model=llm)

agent_response=agent.invoke({
    "messages":[HumanMessage(content="are you working?")]
})

agent_response

{'messages': [HumanMessage(content='are you working?', additional_kwargs={}, response_metadata={}, id='6cd91255-f19b-45e0-a22b-9eece327f30d'),
  AIMessage(content='Yes — I’m working. How can I help?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 10, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DgDDbzcS1tphDfH4MkOIt3mgWKMwS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e31db-f336-7bf1-814d-f80c2e095d06-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 14, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [167]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

class TaskDetails(TypedDict, total=False):
    task: str
    priority: int
    completed: bool
    active: bool


def merge_todo_list(existing: list[TaskDetails], new: list[TaskDetails]) -> list[TaskDetails]:
    merged = list(existing or [])

    for new_task in new or []:
        new_task_name = new_task.get("task", "").lower().strip()
        match_index = None

        for index, existing_task in enumerate(merged):
            existing_task_name = existing_task.get("task", "").lower().strip()
            if new_task_name and new_task_name == existing_task_name:
                match_index = index
                break

        if match_index is None:
            merged.append(new_task)
        else:
            merged[match_index] = {**merged[match_index], **new_task}

    for index, task in enumerate(merged, start=1):
        task["priority"] = index
        task["completed"] = bool(task.get("completed", False))
        task["active"] = bool(task.get("active", True))

    return merged


class TodoListExtraction(TypedDict):
    tasks: list[TaskDetails]


class ActionDecision(TypedDict):
    node_name: Literal["add_tasks_to_todo_list", "update_task", "update_status"]


class BaseState(TypedDict, total=False):
    messages: Annotated[list[BaseMessage], add_messages]
    user_input: str
    normalized_input: str
    action: Literal["add_tasks_to_todo_list", "update_task", "update_status"]
    needs_clarification: bool
    todo_list: Annotated[list[TaskDetails], merge_todo_list]
    response: str
    

In [168]:
import json
from langchain.agents import create_agent
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

try:
    agent
except NameError:
    agent = create_agent(model=llm)

def get_user_text(state: BaseState) -> str:
    message_list = state.get("messages", [])
    if message_list:
        return message_list[-1].content
    return state.get("user_input") or state.get("normalized_input") or ""

def agent_json(prompt: str, fallback: dict) -> dict:
    try:
        result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
        messages = result.get("messages", []) if isinstance(result, dict) else []
        content = messages[-1].content if messages else ""

        if isinstance(content, list):
            content = "".join(part.get("text", "") if isinstance(part, dict) else str(part) for part in content)

        content = str(content).strip()
        if content.startswith("```"):
            content = content.strip("`").removeprefix("json").strip()

        return json.loads(content)
    except Exception:
        return fallback

def add_tasks_to_todo_list(state: BaseState) -> Command:
    user_text = get_user_text(state)
    current_todo_list = state.get("todo_list", [])
    current_max_priority = max((task.get("priority", 0) for task in current_todo_list), default=0)
    
    prompt = f"""
Extract a minimal todo list from the user's message.
Return only valid JSON in this exact shape:
{{"tasks": [{{"task": "task text", "priority": 1, "completed": false, "active": true}}]}}

Rules:
- {user_text}, may have maybe more than one tasks
- {user_text}, do not hallucinate breaking down tasks if input defines the tasks clearly
- prioritize these tasks based on the rules
- Return only concrete tasks the user plans to do.
- priority must be an integer from 1 to 50.
- Use 1 for the most important or earliest task.
- completed must always be false.
- active must always be true.

User message:
{user_text}
"""

    result = agent_json(prompt, {"tasks": []})
    todo_list = result.get("tasks", [])

    if not todo_list and user_text:
        todo_list = [{"task": user_text, "priority": 1, "completed": False, "active": True}]

    todo_list = sorted(todo_list, key=lambda task: task.get("priority", 50))
    for index, task in enumerate(todo_list, start=1):
        task["priority"] = current_max_priority + index
        task["completed"] = bool(task.get("completed", False))
        task["active"] = True

    return Command(
        update={
            "todo_list": todo_list,
            "response": "Initial todo list created.",
        }
    )

def update_task(state: BaseState) -> Command:
    user_text = get_user_text(state)
    todo_list = state.get("todo_list", [])

    prompt = f"""
Find the proper task or tasks to update, then return the full updated todo list.
Return only valid JSON in this exact shape:
{{"tasks": [{{"task": "task text", "priority": 1, "completed": false, "active": true}}]}}

Rules:
- Keep tasks that are not mentioned unchanged.
- Update task text or priority only when the user asks for it.
- Do not invent new tasks.
- priority must be an integer from 1 to 50.
- completed must stay true or false.
- active must stay true or false.

Current todo list:
{todo_list}

User message:
{user_text}
"""

    result = agent_json(prompt, {"tasks": todo_list})
    updated_todo_list = result.get("tasks", todo_list)

    return Command(
        update={
            "todo_list": updated_todo_list,
            "response": "Task updated.",
        }
    )

def update_status(state: BaseState) -> Command:
    user_text = get_user_text(state)
    todo_list = state.get("todo_list", [])

    prompt = f"""
Find the proper task or tasks to update completed/active status, then return only the changed task objects.
Return only valid JSON in this exact shape:
{{"tasks": [{{"task": "task text", "priority": 1, "completed": true, "active": true}}]}}

Rules:
- Only change completed or active status.
- completed true means done, completed, finished, or complete.
- completed false means not done, pending, incomplete, reopen, or undo.
- active false means remove, delete, archive, inactive, or no longer needed.
- active true means restore, reactivate, active, or bring back.
- For remove/delete requests, return the existing matching task text exactly, with active false.
- Keep task text and priority unchanged.
- Do not invent new tasks.

Current todo list:
{todo_list}

User message:
{user_text}
"""

    result = agent_json(prompt, {"tasks": todo_list})
    updated_todo_list = result.get("tasks", todo_list)

    return Command(
        update={
            "todo_list": updated_todo_list,
            "response": "Task status updated.",
        }
    )

def decide_action(state: BaseState) -> Command:
    user_text = get_user_text(state)

    node_list = "add_tasks_to_todo_list, update_task, update_status"
    node_details = """
add_tasks_to_todo_list: use when the user gives a new task list or asks to create/add tasks.
update_task: use when the user wants to edit task text or priority.
update_status: use when the user says a task is done, completed, pending, reopened, not done, removed, deleted, archived, inactive, restored, or active.
"""

    prompt = f"""
Get the user input and decide which node to route to.
Return only valid JSON in this exact shape:
{{"node_name": "add_tasks_to_todo_list"}}

Current node list:
{node_list}

Current node details:
{node_details}

User input:
{user_text}
"""

    result = agent_json(prompt, {})
    node_name = result.get("node_name")

    if not node_name:
        lowered_text = user_text.lower()
        if any(word in lowered_text for word in ("done", "complete", "completed", "finished", "pending", "not done", "reopen", "remove", "delete", "archive", "inactive", "restore", "reactivate")):
            node_name = "update_status"
        elif any(word in lowered_text for word in ("update", "change", "edit", "priority", "prioritize")):
            node_name = "update_task"
        else:
            node_name = "add_tasks_to_todo_list"

    if node_name not in {"add_tasks_to_todo_list", "update_task", "update_status"}:
        node_name = "add_tasks_to_todo_list"

    return Command(update={"action": node_name}, goto=node_name)


In [169]:
import getpass
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
logged_in_user = getpass.getuser()
thread_config = {"configurable": {"thread_id": f"todo-assistant-{logged_in_user}"}}

graph = StateGraph(BaseState)

graph.add_node("decide_action", decide_action)
graph.add_node("add_tasks_to_todo_list", add_tasks_to_todo_list)
graph.add_node("update_task", update_task)
graph.add_node("update_status", update_status)

graph.add_edge(START, "decide_action")

graph.add_edge("add_tasks_to_todo_list", END)
graph.add_edge("update_task", END)
graph.add_edge("update_status", END)

compiled_graph = graph.compile(checkpointer=memory)


In [170]:
from pprint import pprint

latest_final_transcript = """
Final: Today I need to develop the LangGraph voice assistant, test the realtime transcription,
write the todo graph logic, and update the project notes.
"""

state = compiled_graph.invoke(
    {
        "messages": [HumanMessage(content=latest_final_transcript)],
        "todo_list": []
    },
    config=thread_config,
)

pprint(state)

{'action': 'add_tasks_to_todo_list',
 'messages': [HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='24984c08-8a88-4e29-b268-70e84f0859f3')],
 'response': 'Initial todo list created.',
 'todo_list': [{'active': True,
                'completed': False,
                'priority': 1,
                'task': 'Develop the LangGraph voice assistant'},
               {'active': True,
                'completed': False,
                'priority': 2,
                'task': 'Test the realtime transcription'},
               {'active': True,
                'completed': False,
                'priority': 3,
                'task': 'Write the todo graph logic'},
               {'active': True,
                'completed': False,
                'priority': 4,
                'task': 'Update the project notes'}]}


In [92]:
latest_final_transcript = """
Final: Update the LangGraph voice assistant task priority to 43.
"""

state = compiled_graph.invoke(
    {
        "messages": [HumanMessage(content=latest_final_transcript)]
    },
    config=thread_config,
)

pprint(state)

{'action': 'update_task',
 'messages': [HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='3ff7be75-14d8-4404-a1e5-de5286c1ba24'),
              HumanMessage(content='\nFinal: Update the LangGraph voice assistant task priority to 20.\n', additional_kwargs={}, response_metadata={}, id='e51b7fdc-7b51-44c2-a57d-ea07e9243865'),
              HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='30dbc3c5-398c-4698-bcb9-6ac69df159f3'),
              HumanMessage(content='\nFinal: Update the LangGraph voice assistant task priority to 20.\n', additional_kwargs={}, response_metadata={}, id='2929497f-c2ff-4932-948b-5c68a716b5b0'),
              Hu

In [96]:
latest_final_transcript = """
Final: Change the project notes task to write detailed architecture notes.
"""

state = compiled_graph.invoke({
    "messages": [HumanMessage(content=latest_final_transcript)]
},    config=thread_config,
)

pprint(state)

{'action': 'update_task',
 'messages': [HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='3ff7be75-14d8-4404-a1e5-de5286c1ba24'),
              HumanMessage(content='\nFinal: Update the LangGraph voice assistant task priority to 20.\n', additional_kwargs={}, response_metadata={}, id='e51b7fdc-7b51-44c2-a57d-ea07e9243865'),
              HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='30dbc3c5-398c-4698-bcb9-6ac69df159f3'),
              HumanMessage(content='\nFinal: Update the LangGraph voice assistant task priority to 20.\n', additional_kwargs={}, response_metadata={}, id='2929497f-c2ff-4932-948b-5c68a716b5b0'),
              Hu

In [171]:
latest_final_transcript = """
Final: Add these tasks: create a README, prepare the demo script, and clean the notebook.
"""

state = compiled_graph.invoke({
    "messages": [HumanMessage(content=latest_final_transcript)]
},  config=thread_config,)

pprint(state)

{'action': 'add_tasks_to_todo_list',
 'messages': [HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='24984c08-8a88-4e29-b268-70e84f0859f3'),
              HumanMessage(content='\nFinal: Add these tasks: create a README, prepare the demo script, and clean the notebook.\n', additional_kwargs={}, response_metadata={}, id='b238a3bf-be70-4acf-aa30-0db733361095')],
 'response': 'Initial todo list created.',
 'todo_list': [{'active': True,
                'completed': False,
                'priority': 1,
                'task': 'Develop the LangGraph voice assistant'},
               {'active': True,
                'completed': False,
                'priority': 2,
                'task': 'Test the realtime transcription'},
               {'active': True,
                'completed': False,
                'priori

In [172]:
latest_final_transcript = """
remove the task of project note updating task.
"""

state = compiled_graph.invoke(
    {"messages": [HumanMessage(content=latest_final_transcript)]},
    config=thread_config,
)

pprint(state)

{'action': 'update_status',
 'messages': [HumanMessage(content='\nFinal: Today I need to develop the LangGraph voice assistant, test the realtime transcription,\nwrite the todo graph logic, and update the project notes.\n', additional_kwargs={}, response_metadata={}, id='24984c08-8a88-4e29-b268-70e84f0859f3'),
              HumanMessage(content='\nFinal: Add these tasks: create a README, prepare the demo script, and clean the notebook.\n', additional_kwargs={}, response_metadata={}, id='b238a3bf-be70-4acf-aa30-0db733361095'),
              HumanMessage(content='\nremove the task of project note updating task.\n', additional_kwargs={}, response_metadata={}, id='9132eae3-3ea9-4366-b7b3-4851dbacae88')],
 'response': 'Task status updated.',
 'todo_list': [{'active': True,
                'completed': False,
                'priority': 1,
                'task': 'Develop the LangGraph voice assistant'},
               {'active': True,
                'completed': False,
                'pri